[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/03_advanced_architectures/resnet_efficientnet.ipynb)


# 03 . ResNet and EfficientNet Building Blocks

## Concept — Residual Networks & EfficientNet blocks (revision notes)

**What it is**
- A residual block computes `y = x + F(x)`: the layers learn a *correction* to the input rather than a whole new representation. The `+ x` path is the skip (shortcut) connection.
- EfficientNet's MBConv block is an inverted bottleneck: expand channels with 1x1, depthwise 3x3, Squeeze-and-Excitation (channel attention), project back with 1x1, plus a skip when shapes match.
- Compound scaling grows depth, width and resolution together with fixed ratios instead of scaling just one.

**Why it matters**
- Skip connections let you train networks with tens or hundreds of layers; without them, deeper plain nets train WORSE (the degradation problem).
- Depthwise-separable convolutions + SE give much better accuracy per parameter/FLOP, which matters on phones and edge devices.

**When to use**
- ResNet-style blocks are the default safe choice for conv backbones and appear inside nearly every deep architecture (Transformers use the same `x + f(x)` pattern).
- MBConv/EfficientNet when parameter or FLOP budget matters; plain ResNet when simplicity and hardware efficiency (dense convs) matter more.

**Math & intuition**
- Backward through `y = x + F(x)`: `dL/dx = dL/dy * (1 + dF/dx)`. The `1` is an identity highway for gradients, so they cannot vanish multiplicatively through depth (section 3).
- If the best function is the identity, the block only needs `F -> 0`, which is easy; a plain stack has to learn the identity with many weights.
- Depthwise-separable conv params: `k*k*C_in + C_in*C_out` versus `k*k*C_in*C_out` for a regular conv.
- SE: global-average-pool -> bottleneck MLP -> sigmoid -> rescale each channel (learned channel attention).

### 1. A basic residual block

**What this cell does (plain language):** we write the classic ResNet 'BasicBlock': two 3x3 convs with BatchNorm, a ReLU in between, then add the input and apply a final ReLU. When the output shape equals the input shape the shortcut is the identity, so the block adds zero extra parameters for it.

In [1]:
import math, time, torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

class BasicBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(c)
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))          # F(x): note no ReLU yet
        return F.relu(out + x)                   # add the skip, THEN activate

blk = BasicBlock(8)
x = torch.randn(2, 8, 16, 16)
print("in/out shapes:", x.shape, blk(x).shape, "| params:", sum(p.numel() for p in blk.parameters()))


in/out shapes: torch.Size([2, 8, 16, 16]) torch.Size([2, 8, 16, 16]) | params: 1184


### 2. Downsampling block with a projection shortcut

**What this cell does (plain language):** when a block halves the resolution or changes channels, `x` and `F(x)` no longer have the same shape, so the shortcut needs a 1x1 strided convolution to match. We build that variant and print the shapes.

In [2]:
class DownBlock(nn.Module):
    def __init__(self, cin, cout, stride=2):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        # projection shortcut: 1x1 conv with the same stride so spatial size and channels both match
        self.short = nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
    def forward(self, x):
        out = self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
        return F.relu(out + self.short(x))

d = DownBlock(8, 16)
print(x.shape, "->", d(x).shape)
try:
    (x[:, :, ::2, ::2] + torch.randn(2, 16, 8, 8))     # what happens WITHOUT a projection
except RuntimeError as e:
    print("adding mismatched shapes fails:", str(e)[:70])


torch.Size([2, 8, 16, 16]) -> torch.Size([2, 16, 8, 8])
adding mismatched shapes fails: The size of tensor a (8) must match the size of tensor b (16) at non-s


### 3. Gradient flow: plain deep stack vs residual stack

**What this cell does (plain language):** we build a 40-layer MLP twice with PyTorch's default initialisation: once plain (`x = relu(layer(x))`) and once residual (`x = x + relu(layer(x)) / sqrt(depth)`; without normalisation layers the branch must be damped, otherwise 40 un-normalised additions can grow the signal and the gradient instead). After one backward pass we print the gradient norm of the FIRST layer's weights and of the LAST layer. In the plain net the early gradient is vanishingly small compared to the last layer; the residual net keeps gradients healthy through depth.

In [3]:
class Deep(nn.Module):
    def __init__(self, depth=40, w=32, residual=False):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(w, w) for _ in range(depth)]); self.residual = residual
        self.out = nn.Linear(w, 1); self.scale = 1 / math.sqrt(depth)   # damp each residual branch so 40 un-normalised blocks don't blow up
    def forward(self, x):
        for l in self.layers:
            x = x + self.scale * F.relu(l(x)) if self.residual else F.relu(l(x))
        return self.out(x)

torch.manual_seed(0); X = torch.randn(64, 32); Y = torch.randn(64, 1)
for res in [False, True]:
    torch.manual_seed(1); m = Deep(residual=res)
    F.mse_loss(m(X), Y).backward()
    g = [l.weight.grad.norm().item() for l in m.layers]
    print(f"residual={res!s:5} | grad norm first layer {g[0]:.2e} | middle {g[20]:.2e} | last layer {g[-1]:.2e} | first/last ratio {g[0]/g[-1]:.2e}")


residual=False | grad norm first layer 8.93e-17 | middle 8.55e-11 | last layer 6.48e-03 | first/last ratio 1.38e-14
residual=True  | grad norm first layer 1.79e-01 | middle 2.23e-01 | last layer 5.68e-01 | first/last ratio 3.16e-01


### 4. Plot gradient norm per layer

**What this cell does (plain language):** the same experiment as a picture on a log scale: gradient norm of every layer's weights, from first (left) to last (right). The plain stack decays exponentially toward the early layers; the residual stack is comparatively flat.

In [4]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
for res in [False, True]:
    torch.manual_seed(1); m = Deep(residual=res)
    F.mse_loss(m(X), Y).backward()
    ax.semilogy([l.weight.grad.norm().item() for l in m.layers], label="residual" if res else "plain")
ax.set_xlabel("layer index (0 = closest to input)"); ax.set_ylabel("grad norm (log)"); ax.legend(); ax.set_title("Gradient norm vs depth")
plt.tight_layout(); plt.show()


### 5. Degradation: does depth hurt training without skips?

**What this cell does (plain language):** we train both 40-layer MLPs on a small synthetic regression task with the same optimizer and steps, and compare final TRAIN loss. This is the degradation problem: the deeper plain network is not just overfitting, it is harder to optimise.

In [5]:
torch.manual_seed(0)
Xs = torch.randn(256, 32); true_w = torch.randn(32, 1)
Ys = torch.sin(Xs @ true_w) + 0.05 * torch.randn(256, 1)       # nonlinear target
for res in [False, True]:
    torch.manual_seed(1); m = Deep(residual=res); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for step in range(200):
        loss = F.mse_loss(m(Xs), Ys); opt.zero_grad(); loss.backward(); opt.step()
    print(f"residual={res!s:5} | train MSE after 200 steps: {loss.item():.4f}   (predicting the mean would give ~{Ys.var().item():.3f})")


residual=False | train MSE after 200 steps: 0.4725   (predicting the mean would give ~0.474)


residual=True  | train MSE after 200 steps: 0.0000   (predicting the mean would give ~0.474)


### 6. Synthetic image task for the conv nets

**What this cell does (plain language):** we generate 16x16 images of three classes of striped patterns (horizontal grating, vertical grating, checkerboard) with random frequency, phase and noise, entirely in-memory and vectorised. Classes are defined by orientation/structure, not location.

In [6]:
def make_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, 3, (n,), generator=g)
    ii, jj = torch.meshgrid(torch.arange(16.), torch.arange(16.), indexing="ij")
    freq = 0.5 + 1.0 * torch.rand(n, 1, 1, generator=g); ph = 6.28 * torch.rand(n, 1, 1, generator=g)
    h = torch.sin(freq * ii + ph); v = torch.sin(freq * jj + ph); c = h * v      # horizontal / vertical / checker
    x = torch.where((y == 0)[:, None, None], h, torch.where((y == 1)[:, None, None], v, c))
    x = x + 0.3 * torch.randn(n, 16, 16, generator=g)
    return x.unsqueeze(1), y
Xtr, ytr = make_data(900, 0); Xte, yte = make_data(300, 1)
print(Xtr.shape, ytr.bincount().tolist())

def fit(model, epochs=6, lr=3e-3, bs=64):
    torch.manual_seed(0); opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    for ep in range(epochs):
        model.train(); perm = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), bs):
            idx = perm[i:i + bs]; loss = F.cross_entropy(model(Xtr[idx]), ytr[idx]); opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad(): return (model(Xte).argmax(1) == yte).float().mean().item()
nparams = lambda m: sum(p.numel() for p in m.parameters())


torch.Size([900, 1, 16, 16]) [288, 313, 299]


### 7. A mini ResNet and its baseline without skips

**What this cell does (plain language):** we stack residual blocks into a small ResNet (stem -> blocks -> downsample -> global pool -> linear). A `use_skip=False` flag gives the identical network with the skip removed, so any difference comes only from the connection. At this tiny depth both usually train; the skip's advantage grows with depth (previous sections).

In [7]:
class Block(nn.Module):
    def __init__(self, c, use_skip=True):
        super().__init__(); self.use_skip = use_skip
        self.c1 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.b1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1, bias=False); self.b2 = nn.BatchNorm2d(c)
    def forward(self, x):
        out = self.b2(self.c2(F.relu(self.b1(self.c1(x)))))
        return F.relu(out + x) if self.use_skip else F.relu(out)

class MiniResNet(nn.Module):
    def __init__(self, c=16, n_blocks=3, use_skip=True):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU())
        self.blocks = nn.Sequential(*[Block(c, use_skip) for _ in range(n_blocks)])
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, 3))
    def forward(self, x): return self.head(self.blocks(self.stem(x)))

t0 = time.time()
torch.manual_seed(0); res_net = MiniResNet(); acc_res = fit(res_net)
torch.manual_seed(0); plain_net = MiniResNet(use_skip=False); acc_plain = fit(plain_net)
print(f"MiniResNet params {nparams(res_net):,} | acc with skips {acc_res:.3f} | without skips {acc_plain:.3f} | {time.time()-t0:.1f}s")


MiniResNet params 14,243 | acc with skips 0.990 | without skips 1.000 | 14.0s


### 8. Squeeze-and-Excitation block

**What this cell does (plain language):** SE gives the network cheap channel attention: 'squeeze' each channel to one number by global average pooling, 'excite' with a small bottleneck MLP ending in a sigmoid, then multiply each channel by its gate in (0,1). It adds few parameters and no spatial mixing.

In [8]:
class SE(nn.Module):
    def __init__(self, c, reduction=4):
        super().__init__()
        hidden = max(1, c // reduction)
        self.fc1, self.fc2 = nn.Conv2d(c, hidden, 1), nn.Conv2d(hidden, c, 1)    # 1x1 convs act as Linear on the pooled vector
    def forward(self, x):
        s = x.mean(dim=(2, 3), keepdim=True)                  # squeeze: (B,C,1,1) global average pool
        gate = torch.sigmoid(self.fc2(F.silu(self.fc1(s))))   # excite: per-channel gate in (0,1)
        self.last_gate = gate.detach()
        return x * gate                                        # rescale channels

torch.manual_seed(0)
se = SE(16); x = torch.randn(2, 16, 8, 8)
print("shape kept:", se(x).shape, "| gate range:", round(se.last_gate.min().item(), 3), "-", round(se.last_gate.max().item(), 3), "| params:", nparams(se))


shape kept: torch.Size([2, 16, 8, 8]) | gate range: 0.386 - 0.637 | params: 148


### 9. Depthwise-separable convolution: parameter savings

**What this cell does (plain language):** a regular 3x3 conv mixes space and channels at once. A depthwise conv (`groups=C`) filters each channel separately, and a 1x1 pointwise conv then mixes channels. We count parameters and verify the formulas from the concept notes.

In [9]:
cin, cout, k = 64, 128, 3
regular = nn.Conv2d(cin, cout, k, padding=1, bias=False)
dw = nn.Conv2d(cin, cin, k, padding=1, groups=cin, bias=False)     # one k x k filter per input channel
pw = nn.Conv2d(cin, cout, 1, bias=False)                           # channel mixing
sep = nn.Sequential(dw, pw)
print("regular conv params:", nparams(regular), "= k*k*Cin*Cout =", k * k * cin * cout)
print("depthwise+pointwise:", nparams(sep), "= k*k*Cin + Cin*Cout =", k * k * cin + cin * cout)
print(f"reduction: {nparams(regular)/nparams(sep):.1f}x fewer parameters")
x = torch.randn(1, cin, 16, 16); print("same output shape:", regular(x).shape == sep(x).shape)


regular conv params: 73728 = k*k*Cin*Cout = 73728
depthwise+pointwise: 8768 = k*k*Cin + Cin*Cout = 8768
reduction: 8.4x fewer parameters
same output shape: True


### 10. MBConv block with stochastic depth

**What this cell does (plain language):** we assemble EfficientNet's MBConv: 1x1 expand (x`expand`) -> BN+SiLU -> depthwise 3x3 -> BN+SiLU -> SE -> 1x1 project (NO activation: a linear bottleneck) -> BN, plus the residual only when stride is 1 and channels match. Stochastic depth randomly drops the whole residual branch per sample during training.

In [10]:
class MBConv(nn.Module):
    def __init__(self, cin, cout, expand=4, stride=1, drop_path=0.0):
        super().__init__()
        mid = cin * expand
        self.use_skip = (stride == 1 and cin == cout); self.drop_path = drop_path
        layers = []
        if expand != 1: layers += [nn.Conv2d(cin, mid, 1, bias=False), nn.BatchNorm2d(mid), nn.SiLU()]      # expand
        layers += [nn.Conv2d(mid, mid, 3, stride, 1, groups=mid, bias=False), nn.BatchNorm2d(mid), nn.SiLU(),  # depthwise
                   SE(mid, reduction=4 * expand),                                                          # channel attention
                   nn.Conv2d(mid, cout, 1, bias=False), nn.BatchNorm2d(cout)]                              # linear projection
        self.f = nn.Sequential(*layers)
    def forward(self, x):
        out = self.f(x)
        if not self.use_skip: return out
        if self.training and self.drop_path > 0:
            keep = (torch.rand(x.size(0), 1, 1, 1) >= self.drop_path).float() / (1 - self.drop_path)   # per-sample drop, rescaled
            out = out * keep
        return x + out

torch.manual_seed(0)
for cfg in [(16, 16, 4, 1), (16, 24, 4, 2)]:
    m = MBConv(*cfg).eval(); x = torch.randn(2, cfg[0], 16, 16)
    print(f"cin,cout,expand,stride={cfg} -> out {tuple(m(x).shape)} | skip={m.use_skip} | params={nparams(m)}")


cin,cout,expand,stride=(16, 16, 4, 1) -> out (2, 16, 16, 16) | skip=True | params=3492
cin,cout,expand,stride=(16, 24, 4, 2) -> out (2, 24, 8, 8) | skip=False | params=4020


### 11. A mini EfficientNet-style network vs the mini ResNet

**What this cell does (plain language):** we stack a stem, a few MBConv blocks (one downsampling) and a head, then train it on the same data. We report parameter counts side by side. On such an easy toy task accuracy mostly saturates, so the take-away is the parameter efficiency of MBConv rather than a leaderboard.

In [11]:
class MiniEffNet(nn.Module):
    def __init__(self, drop_path=0.1):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8), nn.SiLU())
        self.blocks = nn.Sequential(MBConv(8, 8, 1, 1), MBConv(8, 16, 4, 2, drop_path), MBConv(16, 16, 4, 1, drop_path), MBConv(16, 24, 4, 2, drop_path))
        self.head = nn.Sequential(nn.Conv2d(24, 48, 1), nn.SiLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(48, 3))
    def forward(self, x): return self.head(self.blocks(self.stem(x)))
t0 = time.time(); torch.manual_seed(0); eff = MiniEffNet(); acc_eff = fit(eff)
print(f"MiniEffNet params {nparams(eff):,} acc {acc_eff:.3f} | MiniResNet params {nparams(res_net):,} acc {acc_res:.3f} | {time.time()-t0:.1f}s")


MiniEffNet params 10,535 acc 1.000 | MiniResNet params 14,243 acc 0.990 | 10.1s


### 12. Compound scaling arithmetic

**What this cell does (plain language):** EfficientNet scales depth by `a^phi`, width by `b^phi` and resolution by `c^phi`. FLOPs grow roughly with `depth * width^2 * resolution^2`, so the paper constrains `a * b^2 * c^2 ~ 2`: each unit of phi roughly doubles compute. We tabulate the multipliers using the published B0 coefficients (a=1.2, b=1.1, c=1.15).

In [12]:
a, b, c = 1.2, 1.1, 1.15
print("a*b^2*c^2 =", round(a * b**2 * c**2, 3), "(~2 -> each phi step ~doubles FLOPs)")
for phi in range(0, 5):
    d, w, r = a**phi, b**phi, c**phi
    print(f"phi={phi} | depth x{d:.2f} | width x{w:.2f} | resolution x{r:.2f} | relative FLOPs ~ x{d * w**2 * r**2:.2f}")


a*b^2*c^2 = 1.92 (~2 -> each phi step ~doubles FLOPs)
phi=0 | depth x1.00 | width x1.00 | resolution x1.00 | relative FLOPs ~ x1.00
phi=1 | depth x1.20 | width x1.10 | resolution x1.15 | relative FLOPs ~ x1.92
phi=2 | depth x1.44 | width x1.21 | resolution x1.32 | relative FLOPs ~ x3.69
phi=3 | depth x1.73 | width x1.33 | resolution x1.52 | relative FLOPs ~ x7.08
phi=4 | depth x2.07 | width x1.46 | resolution x1.75 | relative FLOPs ~ x13.60


## Common bugs

- **Shape mismatch on the skip** — `x + F(x)` fails when stride or channels change. Fix: use a 1x1 (strided) projection shortcut, as in section 2.
- **ReLU after the add vs before** — putting ReLU on the residual branch's last layer clips the correction to be non-negative. Fix: BN last on the branch, then add, then activate (or use pre-activation ordering consistently).
- **Skip connection on a downsampling MBConv** — adding `x` to a different-shaped output crashes or silently broadcasts wrongly. Fix: only use the skip when `stride == 1 and cin == cout`.
- **Activation after the MBConv projection** — the 1x1 projection is a *linear* bottleneck; ReLU there destroys information in the low-dimensional space. Fix: no activation after projection.
- **`bias=True` before BatchNorm** — BN's shift makes the conv bias redundant and wastes parameters. Fix: `bias=False` on convs followed by BN.
- **Depthwise conv with the wrong `groups`** — `groups` must equal `in_channels` (and `out_channels` a multiple of it). Fix: `groups=C`.
- **Stochastic depth at eval time or without rescaling** — dropping branches at inference, or forgetting `/(1-p)`, shifts activation scale. Fix: only drop when `self.training` and rescale.
- **Small batches with BatchNorm** — noisy statistics hurt training and eval differences. Fix: larger batches, GroupNorm, or freeze BN stats when fine-tuning.

## Exercises

**Exercise 1.** Implement a ResNet 'bottleneck' block (1x1 reduce to C/4, 3x3, 1x1 expand back to C, with skip) and compare its parameter count to `BasicBlock(C)` for C=64.

In [13]:
# Your attempt for Exercise 1 here


**Solution 1**

In [14]:
class Bottleneck(nn.Module):
    def __init__(self, c, r=4):
        super().__init__(); m = c // r
        self.f = nn.Sequential(nn.Conv2d(c, m, 1, bias=False), nn.BatchNorm2d(m), nn.ReLU(),
                               nn.Conv2d(m, m, 3, padding=1, bias=False), nn.BatchNorm2d(m), nn.ReLU(),
                               nn.Conv2d(m, c, 1, bias=False), nn.BatchNorm2d(c))
    def forward(self, x): return F.relu(self.f(x) + x)
print("Bottleneck(64):", nparams(Bottleneck(64)), "| BasicBlock(64):", nparams(BasicBlock(64)))


Bottleneck(64): 4544 | BasicBlock(64): 73984


**Exercise 2.** Zero-initialise the last BatchNorm gamma in every residual branch of `MiniResNet` (so each block starts as identity) and confirm the network output at init equals the stem+head output on the identity path.

In [15]:
# Your attempt for Exercise 2 here


**Solution 2**

In [16]:
torch.manual_seed(0); net = MiniResNet().eval()
for b in net.blocks: nn.init.zeros_(b.b2.weight); nn.init.zeros_(b.b2.bias)
x = Xte[:4]
with torch.no_grad():
    h = net.stem(x); out_blocks = net.blocks(h)
    print("blocks are identity (up to the final ReLU, h>=0 already):", torch.allclose(out_blocks, h, atol=1e-6))


blocks are identity (up to the final ReLU, h>=0 already): True


**Exercise 3.** Measure the gradient norm of the first conv in `plain_net` vs `res_net` after one backward pass on a batch.

In [17]:
# Your attempt for Exercise 3 here


**Solution 3**

In [18]:
for name, net in [("plain", plain_net), ("residual", res_net)]:
    net.train(); net.zero_grad(); F.cross_entropy(net(Xtr[:64]), ytr[:64]).backward()
    print(name, "stem conv grad norm:", round(net.stem[0].weight.grad.norm().item(), 4))


plain stem conv grad norm: 0.066


residual stem conv grad norm: 0.0573


**Exercise 4.** Increase `Deep` depth to 80 for the plain and residual variants and report the first-layer gradient norm. What happens to the residual one?

In [19]:
# Your attempt for Exercise 4 here


**Solution 4**

In [20]:
for res in [False, True]:
    torch.manual_seed(1); m = Deep(depth=80, residual=res); F.mse_loss(m(X), Y).backward()
    print("residual" if res else "plain   ", f"first-layer grad norm {m.layers[0].weight.grad.norm().item():.2e}")
print("Note: the branch is damped by 1/sqrt(depth); without damping or normalisation a residual stack can explode instead of vanish.")


plain    first-layer grad norm 0.00e+00
residual first-layer grad norm 1.41e-01
Note: the branch is damped by 1/sqrt(depth); without damping or normalisation a residual stack can explode instead of vanish.


**Exercise 5.** Count how many parameters of `MiniEffNet` live inside SE modules versus the whole model.

In [21]:
# Your attempt for Exercise 5 here


**Solution 5**

In [22]:
se_p = sum(nparams(m) for m in eff.modules() if isinstance(m, SE))
print(f"SE params: {se_p} of {nparams(eff)} ({100*se_p/nparams(eff):.1f}%)")


SE params: 1364 of 10535 (12.9%)


**Exercise 6.** Verify that `MBConv` with `expand=1, stride=1, cin==cout` uses a skip, and that in eval mode the drop_path setting has no effect.

In [23]:
# Your attempt for Exercise 6 here


**Solution 6**

In [24]:
m = MBConv(8, 8, 1, 1, drop_path=0.5).eval(); x = torch.randn(2, 8, 8, 8)
with torch.no_grad(): print("skip:", m.use_skip, "| deterministic in eval:", torch.equal(m(x), m(x)))


skip: True | deterministic in eval: True
